# Fase 4: Análisis de Datos y KPIs (Analysis)
En esta fase final, realizaremos el análisis analítico de negocio. Conectaremos directamente a la base de datos local SQLite o a Google BigQuery (según tu configuración), ejecutaremos consultas SQL clave para responder a preguntas estratégicas del negocio, y crearemos visualizaciones claras y dinámicas.

### Consultas Estratégicas a Analizar:
1. **Rendimiento Comercial**: Evolución de ventas y ticket promedio.
2. **Conciliación e Inventario**: Niveles de existencia por bodega.
3. **Rendimiento de Marketing**: ROI y ROAS de campañas por plataforma.
4. **Comportamiento de Clientes**: Clientes principales y segmentación.
5. **Ventas vs. Presupuesto de Marketing**.

In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from dotenv import load_dotenv

sns.set_theme(style="whitegrid")
%matplotlib inline


## 1. Conexión de Datos (Flexibilidad Local / Nube)
Evaluamos la disponibilidad del entorno de BigQuery. Si no está configurado, conectaremos de manera transparente a nuestro SQLite Data Warehouse local.

In [ ]:
load_dotenv()
ID_PROYECTO = os.getenv("ID_PROYECTO")
DW_PATH = "../data/datacommerce_dw.db"

usar_bigquery = False
if ID_PROYECTO:
    try:
        bq_client = bigquery.Client(project=ID_PROYECTO)
        # Validamos acceso
        bq_client.query("SELECT 1;").result()
        usar_bigquery = True
        print(f"✅ Conectado a Google BigQuery exitosamente (Proyecto: {ID_PROYECTO})")
    except Exception as e:
        print(f"⚠️ Error al conectar a BigQuery ({e}). Usaremos el Data Warehouse local SQLite.")
else:
    print("ℹ️ Variable 'ID_PROYECTO' no definida. Usando el Data Warehouse local SQLite.")

def query_dw(sql_sqlite, sql_bq=None):
    """Ejecuta una consulta SQL adaptada a la base de datos activa."""
    if usar_bigquery and sql_bq:
        sql_ejecutar = sql_bq.replace("{dataset}", "proyectofinalG2")
        return bq_client.query(sql_ejecutar).to_dataframe()
    else:
        with sqlite3.connect(DW_PATH) as conn:
            return pd.read_sql_query(sql_sqlite, conn)


## 2. Consulta 1: Rendimiento Comercial
Calculamos las Ventas Totales, el Costo de Ventas, la Utilidad Bruta y el Margen Neto.

In [ ]:
query_comercial = """
SELECT 
    t.anio,
    t.nombre_mes,
    SUM(v.total_venta) as ventas_totales,
    SUM(v.cantidad * p.costo_unitario) as costo_total,
    SUM(v.total_venta - (v.cantidad * p.costo_unitario)) as utilidad_bruta,
    (SUM(v.total_venta - (v.cantidad * p.costo_unitario)) / SUM(v.total_venta)) * 100 as margen_porcentaje
FROM fact_ventas v
JOIN dim_tiempo t ON v.fecha_key = t.fecha_key
JOIN dim_producto p ON v.producto_key = p.producto_key
GROUP BY t.anio, t.nombre_mes, t.mes
ORDER BY t.anio, t.mes;
"""

df_comercial = query_dw(query_comercial, query_comercial)
df_comercial


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df_comercial, x="nombre_mes", y="ventas_totales", palette="viridis")
plt.title("Ventas Totales Mensuales", fontsize=14, fontweight="bold")
plt.xlabel("Mes")
plt.ylabel("Ventas ($)")
plt.tight_layout()
plt.show()


## 3. Consulta 2: ROI / ROAS de Campañas de Marketing
Calculamos la inversión de marketing y métricas agregadas por plataforma.

In [ ]:
query_mkt = """
SELECT 
    c.plataforma,
    SUM(m.costo) as inversion,
    SUM(m.impresiones) as impresiones_totales,
    SUM(m.clics) as clics_totales,
    SUM(m.conversiones) as conversiones_totales
FROM fact_marketing m
JOIN dim_campana c ON m.campana_key = c.campana_key
GROUP BY c.plataforma;
"""

df_mkt = query_dw(query_mkt, query_mkt)
df_mkt["CPC"] = df_mkt["inversion"] / df_mkt["clics_totales"]
df_mkt


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=df_mkt, x="plataforma", y="inversion", palette="magma")
plt.title("Inversión de Marketing por Plataforma", fontsize=14, fontweight="bold")
plt.xlabel("Plataforma")
plt.ylabel("Inversión ($)")
plt.tight_layout()
plt.show()


## 4. Consulta 3: Comportamiento de Clientes por Segmento
Analizamos la contribución de ingresos según el segmento del cliente.

In [ ]:
query_clientes = """
SELECT 
    c.segmento_cliente,
    COUNT(DISTINCT v.venta_id) as transacciones,
    SUM(v.total_venta) as ventas_totales,
    AVG(v.total_venta) as ticket_promedio
FROM fact_ventas v
JOIN dim_cliente c ON v.cliente_key = c.cliente_key
GROUP BY c.segmento_cliente;
"""

df_clientes_seg = query_dw(query_clientes, query_clientes)
df_clientes_seg


In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(df_clientes_seg["ventas_totales"], labels=df_clientes_seg["segmento_cliente"], autopct='%1.1f%%', colors=sns.color_palette("pastel"))
plt.title("Distribución de Ventas por Segmento de Cliente", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Conclusión
¡Hemos recorrido y analizado de punta a punta el pipeline de datos! A través de estos notebooks, logramos:
1. Extraer datos heterogéneos de diferentes orígenes.
2. Transformar, estandarizar y asegurar la calidad de la información.
3. Modelar los datos usando el esquema de Estrella (Star Schema) y cargarlos a SQLite y BigQuery.
4. Realizar consultas analíticas para resolver dudas estratégicas del negocio y graficar las conclusiones.